In [1]:
import os
import sys
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
from importlib import reload
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_selection import mutual_info_regression
import numpy as np
import statsmodels.api as sm  # module to build a LOWESS model
import piecewise_regression #segmented regression

In [2]:
sys.path.append("../../src")

In [3]:
import main

# Meteobikes

Meteobike data from 27th June 2023

In [4]:
# Define the variable 'data' with the file paths
paths = [
    "../../data/raw_data/ALL-SYSTEMS-2018-06-14.csv",
    "../../data/raw_data/ALL-SYSTEMS-2018-06-19.csv",
    "../../data/raw_data/ALL-SYSTEMS-2019-06-24.csv",
    "../../data/raw_data/ALL-SYSTEMS-2019-06-26.csv",
    "../../data/raw_data/ALL-SYSTEMS-2020-07-07.csv",
    "../../data/raw_data/ALL-SYSTEMS-2020-07-13.csv",
    "../../data/raw_data/ALL-SYSTEMS-2021-06-15.csv",
    "../../data/raw_data/ALL-SYSTEMS-2021-07-02.csv",
    "../../data/raw_data/ALL-SYSTEMS-2022-06-29.csv",
    "../../data/raw_data/ALL-SYSTEMS-2022-07-05.csv",
    "../../data/raw_data/ALL-SYSTEMS-2023-06-14.csv",
    "../../data/raw_data/ALL-SYSTEMS-2023-06-27.csv",
    "../../data/raw_data/ALL-SYSTEMS-2024-06-12.csv",
    "../../data/raw_data/ALL-SYSTEMS-2024-06-18.csv",
    "../../data/raw_data/ALL-SYSTEMS-2025-06-23.csv",
    "../../data/raw_data/ALL-SYSTEMS-2026-06-17.csv"
]


In [5]:
# extract dates from path list

dates = [os.path.basename(path).split(".")[0] for path in paths]

In [ ]:
dates

In [ ]:
for i in paths:
    mbike = pd.read_csv(i)
    mbike['WRTM_temp'] = mbike['Absolute_temperature_degC'] - mbike['Temperature_diff_K']
    print('File:', i)
    print('WRTM_temp',mbike['WRTM_temp'].mean())
    print('Absolute_temperature_degC',mbike['Absolute_temperature_degC'].mean())
    print('Temperature_diff_K',mbike['Temperature_diff_K'].mean())

# If running for the first time

In [9]:
bldgs = gpd.read_parquet('../../data/processed_data/processed_buildings_3D.parquet')
streets = gpd.read_parquet('../../data/processed_data/processed_streets.parquet')
nodes = gpd.read_parquet('../../data/processed_data/processed_nodes.parquet')

In [ ]:
paths

In [ ]:
reload(main)

In [ ]:
for i in paths[5:11]:
    mbike = pd.read_csv(i)
    stn_buffers = main.buffer_stations(mbike, radius=300, input_crs='EPSG:4326', output_crs='EPSG:31468', lat_column = 'Latitude', lon_column = 'Longitude')
    print('Buffer done')
    #stn_buffers = main.neighbourhood_graph_params(bldgs, stn_buffers)
    if 'station_id' not in stn_buffers.columns:
        stn_buffers['station_id'] = stn_buffers.index
    b,s,n = main.select_objects(bldgs, streets, nodes, stn_buffers)
    print('Objects selected')
    stn_buffers = main.aggregate_params_reduced(b,s,n, stn_buffers, three_d=True)
    print('Parameters aggregated')

    # replace all NaNs in stn_buffers['BuAre_sum'] with 0
    stn_buffers['BuAre_sum'] = stn_buffers['BuAre_sum'].fillna(0)
    stn_buffers['BuVol_3D_sum'] = stn_buffers['BuVol_3D_sum'].fillna(0)
    stn_buffers['BuEWA_3D_sum'] = stn_buffers['BuEWA_3D_sum'].fillna(0)

    stn_buffers = stn_buffers[stn_buffers.geometry.notna() & ~stn_buffers.geometry.is_empty].copy()

    masked_svf_path = '../../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')
    stn_buffers.to_parquet('../../data/processed_data/mbike/'+os.path.basename(i).split(".")[0]+'_processed_station_params_300.parquet')

In [ ]:
main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')

In [ ]:
i

In [13]:
bldgs_merged = gpd.read_parquet('../../data/processed_data/processed_buildings_merged_blocks.parquet')

In [ ]:
# for merged blocks
for i in paths[:11]:
    mbike = pd.read_csv(i)
    stn_buffers = main.buffer_stations(mbike, radius=300, input_crs='EPSG:4326', output_crs='EPSG:31468', lat_column = 'Latitude', lon_column = 'Longitude')
    stn_buffers = main.neighbourhood_graph_params(bldgs_merged, stn_buffers)
    b,s,n = main.select_objects(bldgs_merged, streets, nodes, stn_buffers)
    stn_buffers = main.aggregate_params(b,s,n, stn_buffers, three_d=True)

    # replace all NaNs in stn_buffers['BuAre_sum'] with 0
    stn_buffers['BuAre_sum'] = stn_buffers['BuAre_sum'].fillna(0)
    stn_buffers['BuVol_3D_sum'] = stn_buffers['BuVol_3D_sum'].fillna(0)
    stn_buffers['BuEWA_3D_sum'] = stn_buffers['BuEWA_3D_sum'].fillna(0)

    masked_svf_path = '../../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')
    stn_buffers.to_parquet('../../data/processed_data/mbike/'+os.path.basename(i).split(".")[0]+'_processed_station_params_merged_300.parquet')

# Plot making

In [ ]:
var_name_mapping = {
    'BuAre_sum': 'Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)',
    'BuVol_3D_sum': 'Building Volume Density ($\it{λ_V}$, m$^3$ m$^{-2}$)',
    'BuEWA_3D_sum': 'Exposed Facet Density ($\it{λ_E}$, m$^2$ m$^{-2}$)', 
    'BuAdj': 'Adjacency ($\it{Adj}$)',
    'BuHt_wmean': 'Height ($\it{H}$, m)',
    'SVF_3D_mean': 'Sky View Factor ($\it{SVF}$)',
    'BuERI_mode_merged': 'Shape Complexity ($\it{SC}$)',
    'StrClo400_median': 'Street Network Connectivity ($\it{C}$)',
    'StrHW_wmean': 'Height:Width v3 ($\it{H:W_3}$)',
    'HW_geo':'Height:Width v2 ($\it{H:W_2}$)',
    'HW_del_merged':'Height:Width v1 ($\it{H:W_1}$)',
    'BuW_delaunay_mean_merged':'Width ($\it{W}$, m)',
    'BuW_delaunay_mean':'Width (cadastre) ($\it{W}$, m)',
    'HW_knn1_merged':'HW_knn1_merged',
    'HW_knn5_merged':'HW_knn5_merged',
    'NdDen':'$\it{ND}$'
    }

In [ ]:
dfs = {}
for i in paths:
    print(i)
    mbike = pd.read_csv(i)
    stn_buffers = gpd.read_parquet('../../data/processed_data/mbike/'+os.path.basename(i).split(".")[0]+'_processed_station_params_300.parquet')
    stn_buffers_merged = gpd.read_parquet('../../data/processed_data/mbike/'+os.path.basename(i).split(".")[0]+'_processed_station_params_merged_300.parquet')

    stn_buffers['WRTM_temp'] = stn_buffers['Absolute_temperature_degC'] - stn_buffers['Temperature_diff_K']
    stn_buffers['temp_no_cooling'] = stn_buffers['Temperature_diff_K'] + stn_buffers['WRTM_temp'].mean()
    
    radius = 300
    stn_buffers['BuAre_sum'] = stn_buffers['BuAre_sum'] / (radius**2*np.pi)
    stn_buffers['BuVol_3D_sum'] = stn_buffers['BuVol_3D_sum'] / (radius**2*np.pi)
    stn_buffers['BuEWA_3D_sum'] = stn_buffers['BuEWA_3D_sum'] / (radius**2*np.pi)
    stn_buffers['HW_del_merged'] = stn_buffers['BuHt_wmean'] / stn_buffers_merged['BuW_delaunay_mean']
    stn_buffers['BuW_delaunay_mean_merged'] = stn_buffers_merged['BuW_delaunay_mean']
    stn_buffers['BuERI_mode_merged'] = stn_buffers_merged['BuERI_mode']
    stn_buffers['HW_geo'] = stn_buffers['BuEWA_3D_sum'] / (1 - stn_buffers['BuAre_sum'])
    
    dfs[os.path.basename(i).split(".")[0]] = stn_buffers
    plt.scatter(stn_buffers['BuAre_sum'],stn_buffers['temp_no_cooling'])
    plt.grid()
    plt.show()

In [10]:
dfs['ALL-SYSTEMS-2018-06-14'] = dfs['ALL-SYSTEMS-2018-06-14'][dfs['ALL-SYSTEMS-2018-06-14']['temp_no_cooling']<19.5]
dfs['ALL-SYSTEMS-2018-06-19'] = dfs['ALL-SYSTEMS-2018-06-19'][dfs['ALL-SYSTEMS-2018-06-19']['temp_no_cooling']<24.8]
dfs['ALL-SYSTEMS-2019-06-24'] = dfs['ALL-SYSTEMS-2019-06-24'][(dfs['ALL-SYSTEMS-2019-06-24']['temp_no_cooling']<30) & (dfs['ALL-SYSTEMS-2019-06-24']['temp_no_cooling']>20)]
dfs['ALL-SYSTEMS-2019-06-26'] = dfs['ALL-SYSTEMS-2019-06-26'][(dfs['ALL-SYSTEMS-2019-06-26']['temp_no_cooling']<32) & (dfs['ALL-SYSTEMS-2019-06-26']['temp_no_cooling']>22)]
dfs['ALL-SYSTEMS-2020-07-07'] = dfs['ALL-SYSTEMS-2020-07-07'][(dfs['ALL-SYSTEMS-2020-07-07']['temp_no_cooling']<21)]
dfs['ALL-SYSTEMS-2020-07-13'] = dfs['ALL-SYSTEMS-2020-07-13'][(dfs['ALL-SYSTEMS-2020-07-13']['temp_no_cooling']<22.5)]
dfs['ALL-SYSTEMS-2023-06-14'] = dfs['ALL-SYSTEMS-2023-06-14'][(dfs['ALL-SYSTEMS-2023-06-14']['temp_no_cooling']<23.5)]
dfs['ALL-SYSTEMS-2023-06-27'] = dfs['ALL-SYSTEMS-2023-06-27'][(dfs['ALL-SYSTEMS-2023-06-27']['temp_no_cooling']<23)]
dfs['ALL-SYSTEMS-2024-06-12'] = dfs['ALL-SYSTEMS-2024-06-12'][(dfs['ALL-SYSTEMS-2024-06-12']['temp_no_cooling']<18)]
dfs['ALL-SYSTEMS-2025-06-23'] = dfs['ALL-SYSTEMS-2025-06-23'][(dfs['ALL-SYSTEMS-2025-06-23']['temp_no_cooling']<25.5)]
dfs['ALL-SYSTEMS-2026-06-17'] = dfs['ALL-SYSTEMS-2026-06-17'][(dfs['ALL-SYSTEMS-2026-06-17']['temp_no_cooling']>15)]

# Final Segmented Regression Versions

In [ ]:
stn_buffers

In [ ]:
pr = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]

    data= stn_buffers[['BuAre_sum','Temperature_diff_K']].dropna()

    x = data['BuAre_sum'].values
    y = data['Temperature_diff_K'].values + 0.2


    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    #pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
    #pw_fit.plot_breakpoints()
    #pw_fit.plot_breakpoint_confidence_intervals()
plt.xlabel(i)
plt.ylabel("Ta")
plt.show()

In [ ]:
for i in paths:
    pw_fit = pr_area[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)')
plt.ylabel("Ta")
plt.show()   

In [ ]:
for i in paths:
    pw_fit = pr_area[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)')
plt.ylabel("Ta")
plt.show()   

In [ ]:
for i in paths:
    pw_fit = pr_area[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)')
plt.ylabel("Ta")
plt.show()   

In [ ]:
for i in paths:
    pw_fit = pr_vol[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Volume ($\it{λ_P}$, m$^2$ m$^{-2}$)')
plt.ylabel("Ta")
plt.show()   

In [ ]:
for i in paths:
    pw_fit = pr_ewa[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Volume ($\it{λ_P}$, m$^2$ m$^{-2}$)')
plt.ylabel("Ta")
plt.show()   

In [ ]:
pr_area = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]

    data= stn_buffers[['BuAre_sum','Temperature_diff_K']].dropna()

    x = data['BuAre_sum'].values
    y = data['Temperature_diff_K'].values + 0.2

    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pr_area[os.path.basename(i).split(".")[0]] = pw_fit
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
pr_vol = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]

    data= stn_buffers[['BuVol_3D_sum','Temperature_diff_K']].dropna()

    x = data['BuVol_3D_sum'].values
    y = data['Temperature_diff_K'].values + 0.2

    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pr_vol[os.path.basename(i).split(".")[0]] = pw_fit
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
pr_ewa

In [ ]:
pr_ewa = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]

    data= stn_buffers[['BuEWA_3D_sum','Temperature_diff_K']].dropna()

    x = data['BuEWA_3D_sum'].values
    y = data['Temperature_diff_K'].values + 0.2

    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pr_ewa[os.path.basename(i).split(".")[0]] = pw_fit
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
pr_height = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]
    print(i)

    data= stn_buffers[['BuHt_wmean','Temperature_diff_K']].dropna()

    x = data['BuHt_wmean'].values
    y = data['Temperature_diff_K'].values + 0.2

    x = sm.add_constant(x)  # Add a constant term for the intercept
    model = sm.OLS(y, x).fit()
    #bse = model.bse['BuHt_wmean']
    r_squared = model.rsquared

    # Get the predicted values (fitted values)
    y_pred = model.fittedvalues

    # Get the slope params
    slope = model.params[1]
    intercept = model.params[0]
    summary = model.summary()

    print(slope, intercept, r_squared)
    pr_height[os.path.basename(i).split(".")[0]] = model
    plt.plot(x, y_pred, color='grey', linewidth=1)
    plt.text(x[-1][1], y_pred[-1], i, fontsize=10, color='black')
plt.show()

In [ ]:
pr_width = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]
    print(i)

    data= stn_buffers[['BuW_delaunay_mean_merged','Temperature_diff_K']].dropna()

    x = data['BuW_delaunay_mean_merged'].values
    y = data['Temperature_diff_K'].values + 0.2

    x = sm.add_constant(x)  # Add a constant term for the intercept
    model = sm.OLS(y, x).fit()
    #bse = model.bse['BuHt_wmean']
    r_squared = model.rsquared

    # Get the predicted values (fitted values)
    y_pred = model.fittedvalues

    # Get the slope params
    slope = model.params[1]
    intercept = model.params[0]
    summary = model.summary()

    print(slope, intercept, r_squared)
    pr_width[os.path.basename(i).split(".")[0]] = model
    plt.plot(x, y_pred, color='grey', linewidth=1)
    plt.text(x[-1][1], y_pred[-1], i, fontsize=10, color='black')
plt.show()

In [ ]:
x[-1][1]

In [ ]:
pr_svf = {}

for i in paths[0:4]+paths[5:]:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]

    data= stn_buffers[['SVF_3D_mean','Temperature_diff_K']].dropna()

    x = data['SVF_3D_mean'].values
    y = data['Temperature_diff_K'].values + 0.2

    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pr_svf[os.path.basename(i).split(".")[0]] = pw_fit

    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
pr_hw1 = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]

    data= stn_buffers[['HW_del_merged','Temperature_diff_K']].dropna()

    x = data['HW_del_merged'].values
    y = data['Temperature_diff_K'].values + 0.2

    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pr_hw1[os.path.basename(i).split(".")[0]] = pw_fit
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
pr_hw2 = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]

    data= stn_buffers[['HW_geo','Temperature_diff_K']].dropna()

    x = data['HW_geo'].values
    y = data['Temperature_diff_K'].values + 0.2

    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pr_hw2[os.path.basename(i).split(".")[0]] = pw_fit
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
pr_hw3 = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]

    data= stn_buffers[['StrHW_wmean','Temperature_diff_K']].dropna()

    x = data['StrHW_wmean'].values
    y = data['Temperature_diff_K'].values + 0.2

    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pr_hw3[os.path.basename(i).split(".")[0]] = pw_fit
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
pr_eri = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]

    data= stn_buffers[['BuERI_mode_merged','Temperature_diff_K']].dropna()

    x = data['BuERI_mode_merged'].values
    y = data['Temperature_diff_K'].values + 0.2

    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pr_eri[os.path.basename(i).split(".")[0]] = pw_fit
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
pr_eri2 = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]
    print(i)

    data= stn_buffers[['BuERI_mode_merged','Temperature_diff_K']].dropna()

    x = data['BuERI_mode_merged'].values
    y = data['Temperature_diff_K'].values + 0.2

    x = sm.add_constant(x)  # Add a constant term for the intercept
    model = sm.OLS(y, x).fit()
    #bse = model.bse['BuHt_wmean']
    r_squared = model.rsquared

    # Get the predicted values (fitted values)
    y_pred = model.fittedvalues

    # Get the slope params
    slope = model.params[1]
    intercept = model.params[0]
    summary = model.summary()

    print(slope, intercept, r_squared)
    pr_eri2[os.path.basename(i).split(".")[0]] = model
    plt.plot(x, y_pred, color='grey', linewidth=1)
    plt.text(x[-1][1], y_pred[-1], i, fontsize=10, color='black')
plt.show()

In [125]:
list1 = [0.16,	0.17,	0.19,	0.24,	0.18,	0.23,	0.23,	0.22,	0.19,	0.28,	0.17,	0.24,	0.17,	0.16,	0.15,	0.20]
list2 = [-0.0035,	-0.0043,	-0.0038,	-0.0045,	-0.0046,	-0.0060,	-0.0048,	-0.0044,	-0.0044,	-0.0060,	-0.0018,	-0.0041,	-0.0034,	-0.0036,	-0.0040,	-0.0041]
list3 = [-3.96,	-5.53,	-4.86,	-6.21,	-5.06,	-6.52,	-5.51,	-5.11,	-4.88,	-6.99,	-4.99,	-6.30,	-4.64,	-4.06,	-4.72,	-5.78]

In [ ]:
np.median(list3)

In [ ]:
np.median(list2)
np.percentile(list3, 25)

In [ ]:
pr_adj = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]
    print(i)

    data= stn_buffers[['BuAdj','Temperature_diff_K']].dropna()

    x = data['BuAdj'].values
    y = data['Temperature_diff_K'].values + 0.2

    x = sm.add_constant(x)  # Add a constant term for the intercept
    model = sm.OLS(y, x).fit()
    #bse = model.bse['BuHt_wmean']
    r_squared = model.rsquared

    # Get the predicted values (fitted values)
    y_pred = model.fittedvalues

    # Get the slope params
    slope = model.params[1]
    intercept = model.params[0]
    summary = model.summary()

    print(slope, intercept, r_squared)
    pr_adj[os.path.basename(i).split(".")[0]] = model
    plt.plot(x, y_pred, color='grey', linewidth=1)
    plt.text(x[-1][1], y_pred[-1], i, fontsize=10, color='black')
plt.show()

In [ ]:
pr_c = {}

for i in paths:
    stn_buffers = dfs[os.path.basename(i).split(".")[0]]

    data= stn_buffers[['StrClo400_median','Temperature_diff_K']].dropna()

    x = data['StrClo400_median'].values
    y = data['Temperature_diff_K'].values + 0.2

    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pr_c[os.path.basename(i).split(".")[0]] = pw_fit
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

# Multi-plot

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 8))

plt.sca(axes[0, 0])
for i in paths:
    print(i)
    pw_fit = pr_area[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)')
plt.ylabel('$\it{ΔT_{mob}}$ (K)')
plt.ylim(-4,2)
plt.grid()

plt.sca(axes[0, 1])
for i in paths:
    print(i)
    pw_fit = pr_vol[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Building Volume Density ($\it{λ_V}$, m$^3$ m$^{-2}$)')
plt.ylim(-4,2)
plt.grid()

plt.sca(axes[0, 2])
for i in paths:
    print(i)
    pw_fit = pr_ewa[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Exposed Facet Density ($\it{λ_E}$, m$^2$ m$^{-2}$)')
plt.ylim(-4,2)
plt.grid()

plt.sca(axes[0, 3])
for i in paths[0:4]+paths[5:]:
    print(i)
    pw_fit = pr_svf[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Sky View Factor ($\it{SVF}$)')
plt.ylim(-4,2)
plt.grid()

plt.sca(axes[1, 0])
for i in paths:
    print(i)
    pw_fit = pr_hw1[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Height:Width v1 ($\it{H:W_1}$)')
plt.ylabel('$\it{ΔT_{mob}}$ (K)')
plt.ylim(-4,2)
plt.grid()

plt.sca(axes[1, 1])
for i in paths:
    print(i)
    pw_fit = pr_hw2[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Height:Width v2 ($\it{H:W_2}$)')
plt.ylim(-4,2)
plt.grid()

plt.sca(axes[1, 2])
for i in paths:
    print(i)
    pw_fit = pr_hw3[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Height:Width v3 ($\it{H:W_3}$)')
plt.ylim(-4,2)
plt.grid()

plt.sca(axes[1, 3])
for i in paths:
    print(i)
    pw_fit = pr_c[os.path.basename(i).split(".")[0]]
    pw_fit.plot_fit(color="grey", linewidth=1, alpha = 0.5)
plt.xlabel('Street Network Connectivity ($\it{C}$)')
plt.ylim(-4,2)
plt.grid()

plt.savefig('../../figures/fig6/piecewise_reg.svg', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
help(pw_fit.plot_fit)

In [59]:
fit = pw_fit.best_muggeo.best_fit

params = fit.raw_params
bp = fit.next_breakpoints[0]

intercept = params[0]
slope1 = params[1]
beta = params[2]

x_plot = np.linspace(min(x), max(x), 300)

y_plot = (
    intercept
    + slope1 * x_plot
    + beta * np.maximum(x_plot - bp, 0)
)

In [61]:
ax.plot(x_plot, y_plot, linewidth=2)
ax.axvline(bp, linestyle="--")
plt.show()

In [ ]:
print(type(pw_fit))
print(type(pw_fit).__module__)
print(type(pw_fit).__dict__.keys())

In [ ]:
pw_fit.plot_fit()

In [ ]:
pw_fit.xx

In [ ]:
pw_fit.yy

Done

In [ ]:
var_name_mapping = {
    'BuAre_sum': 'Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)',
    'BuVol_3D_sum': 'Building Volume Density ($\it{λ_V}$, m$^3$ m$^{-2}$)',
    'BuEWA_3D_sum': 'Exposed Facet Density ($\it{λ_E}$, m$^2$ m$^{-2}$)', 
    'BuAdj': 'Adjacency ($\it{Adj}$)',
    'BuHt_wmean': 'Height ($\it{H}$, m)',
    'SVF_3D_mean': 'Sky View Factor ($\it{SVF}$)',
    'BuERI_mode_merged': 'Shape Complexity ($\it{SC}$)',
    'StrClo400_median': 'Street Network Connectivity ($\it{C}$)',
    'StrHW_wmean': 'Height:Width v3 ($\it{H:W_3}$)',
    'HW_geo':'Height:Width v2 ($\it{H:W_2}$)',
    'HW_del_merged':'Height:Width v1 ($\it{H:W_1}$)',
    'BuW_delaunay_mean_merged':'Width ($\it{W}$, m)',
    'BuW_delaunay_mean':'Width (cadastre) ($\it{W}$, m)',
    'HW_knn1_merged':'HW_knn1_merged',
    'HW_knn5_merged':'HW_knn5_merged',
    'NdDen':'$\it{ND}$'
    }

In [ ]:
# 2018 data

params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
for i in params:
    print(i)

    if i == 'BuW_delaunay_mean_merged':
        data = stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250][[i,'temp_no_cooling']].dropna()
    else: 
        data= stn_buffers[[i,'temp_no_cooling']].dropna()

    if (i == 'BuERI_mode_merged' or i == 'BuAdj'):
        x = (1-data[i]).values
    else:
        x = data[i].values
    y = data['temp_no_cooling'].values + 0.2


    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
for i in params:
    print(i)

    if i == 'BuW_delaunay_mean_merged':
        data = stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250][[i,'temp_no_cooling']].dropna()
    else: 
        data= stn_buffers[[i,'temp_no_cooling']].dropna()

    if (i == 'BuERI_mode_merged' or i == 'BuAdj'):
        x = (1-data[i]).values
    else:
        x = data[i].values
    y = data['temp_no_cooling'].values + 0.2


    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

print(i)

In [ ]:
# with bootstrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import piecewise_regression

# Configuration
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
n_iterations = 5  # Start with 100, increase to 500-1000 for final paper
system_ids = stn_buffers['System_ID'].unique()

for i in params:
    print(f"Analyzing: {i}")
    
    # --- Bootstrap Loop ---
    boot_breakpoints = []
    
    for b in range(n_iterations):
        # 1. Resample System_IDs with replacement
        resampled_ids = np.random.choice(system_ids, size=len(system_ids), replace=True)
        
        # 2. Filter dataframe to only include these routes
        # This preserves the correlation within each route
        boot_df = stn_buffers[stn_buffers['System_ID'].isin(resampled_ids)].copy()
        
        # Apply your specific parameter filters
        if i == 'BuW_delaunay_mean_merged':
            boot_df = boot_df[boot_df['BuW_delaunay_mean_merged'] < 250]
        
        # Prepare x and y
        data_clean = boot_df[[i, 'temp_no_cooling']].dropna()
        
        if len(data_clean) < 10: # Safety check for empty resamples
            continue
            
        if (i == 'BuERI_mode_merged' or i == 'BuAdj'):
            x_boot = (1 - data_clean[i]).values
        else:
            x_boot = data_clean[i].values
            
        y_boot = data_clean['temp_no_cooling'].values + 0.2

        try:
            # Fit model and extract the breakpoint
            pw_fit_boot = piecewise_regression.Fit(x_boot, y_boot, n_breakpoints=1)
            # Assuming the breakpoint is stored in the model's breakpoints_ attribute
            boot_breakpoints.append(pw_fit_boot.breakpoints[0])
        except:
            # Catch cases where the model fails to converge on a specific resample
            continue

    # Calculate Conservative Confidence Interval (2.5% and 97.5% percentiles)
    ci_low = np.percentile(boot_breakpoints, 2.5)
    ci_high = np.percentile(boot_breakpoints, 97.5)
    median_bp = np.median(boot_breakpoints)
    
    print(f"Bootstrap Breakpoint: {median_bp:.4f} (95% CI: {ci_low:.4f} - {ci_high:.4f})")

    # --- Final Plotting (using the original full dataset) ---
    if i == 'BuW_delaunay_mean_merged':
        data = stn_buffers[stn_buffers['BuW_delaunay_mean_merged'] < 250][[i, 'temp_no_cooling']].dropna()
    else: 
        data = stn_buffers[[i, 'temp_no_cooling']].dropna()

    if (i == 'BuERI_mode_merged' or i == 'BuAdj'):
        x = (1 - data[i]).values
    else:
        x = data[i].values
    y = data['temp_no_cooling'].values + 0.2

    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    
    plt.figure(figsize=(8, 6))
    pw_fit.plot_data(color="grey", s=20)
    pw_fit.plot_fit(color="red", linewidth=4)
    
    # Instead of the default CI, plot the bootstrap median and CI
    plt.axvline(median_bp, color='blue', linestyle='-', label=f'Boot BP: {median_bp:.3f}')
    plt.axvline(ci_low, color='blue', linestyle='--', label=f'95% CI: {ci_low:.3f} - {ci_high:.3f}')
    plt.axvline(ci_high, color='blue', linestyle='--')
    
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.legend()
    plt.show()

In [ ]:
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
for i in params:
    print(i)

    if i == 'BuW_delaunay_mean_merged':
        data = stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<100][[i,'temp_no_cooling']].dropna()

    if i == 'BuHt_wmean':
        data = stn_buffers[stn_buffers['BuHt_wmean']<20][[i,'temp_no_cooling']].dropna()

    else: 
        data= stn_buffers[[i,'temp_no_cooling']].dropna()

    if (i == 'BuERI_mode_merged' or i == 'BuAdj'):
        x = (1-data[i]).values
    

    else:
        x = data[i].values
    y = data['temp_no_cooling'].values + 0.2


    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']
for i in params:
    print(i)

    if i == 'BuW_delaunay_mean_merged':
        data = stn_buffers[stn_buffers['BuW_delaunay_mean_merged']<250][[i,'temp_no_cooling']].dropna()
    else: 
        data= stn_buffers[[i,'temp_no_cooling']].dropna()

    if i = 'BuERI_mode_merged' or i == 'BuAdj':
        x = (1-data[i]).values
    else:
        x = data[i].values
    y = data['temp_no_cooling'].values + 0.2


    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
    pw_fit.summary()

    # Plot the data, fit, breakpoints and confidence intervals
    pw_fit.plot_data(color="grey", s=20)
    # Pass in standard matplotlib keywords to control any of the plots
    pw_fit.plot_fit(color="red", linewidth=4)
    pw_fit.plot_breakpoints()
    pw_fit.plot_breakpoint_confidence_intervals()
    plt.xlabel(i)
    plt.ylabel("Ta")
    plt.show()

In [ ]:
for i in range(12):  # First 12 subplots

    if params[i] == 'BuAre_sum':
        data= stn_buffers[['BuAre_sum','temp_no_cooling']].dropna()

        x = data['BuAre_sum'].values
        y = data['temp_no_cooling'].values + 0.2


        pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=1)
        pw_fit.summary()

        # Plot the data, fit, breakpoints and confidence intervals
        pw_fit.plot_data(color="grey", s=20)
        # Pass in standard matplotlib keywords to control any of the plots
        pw_fit.plot_fit(color="red", linewidth=4)
        pw_fit.plot_breakpoints()
        pw_fit.plot_breakpoint_confidence_intervals()
        plt.xlabel("x")
        plt.ylabel("y")
        plt.show()

In [ ]:
np.isfinite(x)

In [ ]:
plt.scatter(x,y)

In [ ]:
x_normalized

In [ ]:
len(stn_buffers['BuAre_sum'].values)

In [ ]:
len(stn_buffers['temp_no_cooling'].values+0.2)

In [ ]:
stn_buffers['temp_no_cooling'].values